In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import time
import warnings

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_TITLE = "Against All Odds"
WEEK_NUMBER = 2
RANDOM_SEED = 42

NBA_SEASONS = [2021, 2022, 2023, 2024, 2025]
NFL_SEASONS = [2021, 2022, 2023, 2024, 2025]
EPL_SEASONS = {
    "2021-22": "2122",
    "2022-23": "2223",
    "2023-24": "2324",
    "2024-25": "2425",
    "2025-26": "2526",
}

NBA_BASE = "https://api.balldontlie.io"
NFL_BASE = "https://api.balldontlie.io"
FOOTBALL_DATA_BASE = "https://www.football-data.co.uk/mmz4281"

BDL_MIN_INTERVAL_SECONDS = 13.0
REQUEST_TIMEOUT = 30

NBA_CURRENT_ABBRS = {
    "ATL","BOS","BKN","CHA","CHI","CLE","DAL","DEN","DET","GSW",
    "HOU","IND","LAC","LAL","MEM","MIA","MIL","MIN","NOP","NYK",
    "OKC","ORL","PHI","PHX","POR","SAC","SAS","TOR","UTA","WAS"
}

NFL_CURRENT_ABBRS = {
    "ARI","ATL","BAL","BUF","CAR","CHI","CIN","CLE",
    "DAL","DEN","DET","GB","HOU","IND","JAX","KC",
    "LV","LAC","LAR","MIA","MIN","NE","NO","NYG",
    "NYJ","PHI","PIT","SEA","SF","TB","TEN","WAS"
}

print(f"✅ {PROJECT_TITLE} — Week {WEEK_NUMBER} configuration loaded")
print("NBA seasons:", NBA_SEASONS)
print("NFL seasons:", NFL_SEASONS)
print("Premier League seasons:", list(EPL_SEASONS))

# ============================================================
# 2. Locate/create the project directory
# ============================================================

def running_in_colab():
    return "google.colab" in sys.modules

CURRENT_DIR = Path.cwd().resolve()

KNOWN_REPO_NAMES = {
    "against-all-odds",
    "againstallodds",
    "Against-All-Odds",
    "Against_All_Odds",
    "Against All Odds",
}

if CURRENT_DIR.name in KNOWN_REPO_NAMES:
    PROJECT_ROOT = CURRENT_DIR
elif running_in_colab():
    PROJECT_ROOT = Path("/content/againstallodds")
else:
    PROJECT_ROOT = CURRENT_DIR / "againstallodds"

DIRECTORIES = [
    "data/raw/nba",
    "data/raw/nfl",
    "data/raw/soccer",
    "data/clean/nba",
    "data/clean/nfl",
    "data/clean/soccer",
    "data/model_ready/nba",
    "data/model_ready/nfl",
    "data/model_ready/soccer",
    "notebooks",
    "src/collection",
    "src/cleaning",
    "src/features",
    "src/modeling",
    "src/evaluation",
    "figures",
    "results",
    "docs",
    "report",
    "presentation",
]

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for rel in DIRECTORIES:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Verified {len(DIRECTORIES)} project directories")

# ============================================================
# 3. Securely load the BALLDONTLIE API key
# ============================================================

def get_secret(name):
    value = None

    if running_in_colab():
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None

    if not value:
        value = os.getenv(name)

    return value.strip() if isinstance(value, str) and value.strip() else None


BDL_API_KEY = get_secret("BALLDONTLIE_API_KEY")
BDL_HEADERS = {"Authorization": BDL_API_KEY} if BDL_API_KEY else {}

if BDL_API_KEY:
    print("✅ BALLDONTLIE API key loaded securely.")
else:
    print(
        "⚠️ BALLDONTLIE_API_KEY was not found. "
        "NBA/NFL API cells will record SKIPPED results."
    )

# ============================================================
# 4. Reusable request/audit helpers
# ============================================================

_session = requests.Session()
_last_bdl_request_time = 0.0
audit_rows = []


def _rate_limit_bdl():
    global _last_bdl_request_time

    elapsed = time.time() - _last_bdl_request_time
    wait = BDL_MIN_INTERVAL_SECONDS - elapsed

    if wait > 0:
        time.sleep(wait)

    _last_bdl_request_time = time.time()


def safe_get(
    url,
    *,
    params=None,
    headers=None,
    timeout=REQUEST_TIMEOUT,
    use_bdl_rate_limit=False,
    max_retries=3,
):
    # HTTP GET with timeout, retry handling, and optional BDL throttling.
    for attempt in range(1, max_retries + 1):
        try:
            if use_bdl_rate_limit:
                _rate_limit_bdl()

            response = _session.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout,
            )

            if response.status_code == 429 and attempt < max_retries:
                retry_after = response.headers.get("Retry-After")
                sleep_for = float(retry_after) if retry_after else max(
                    BDL_MIN_INTERVAL_SECONDS, 15.0
                )
                print(f"⏳ HTTP 429. Waiting {sleep_for:.0f}s before retry...")
                time.sleep(sleep_for)
                continue

            return response, None

        except requests.RequestException as exc:
            if attempt == max_retries:
                return None, f"{type(exc).__name__}: {exc}"
            time.sleep(2 * attempt)

    return None, "Unknown request failure"


def json_data_to_frame(payload):
    # Convert a BALLDONTLIE-style JSON response to a DataFrame.
    if not isinstance(payload, dict):
        return pd.DataFrame()

    data = payload.get("data", [])

    if isinstance(data, dict):
        data = [data]

    if not isinstance(data, list):
        return pd.DataFrame()

    return pd.json_normalize(data)


def missingness_table(df):
    if df is None or df.empty:
        return pd.DataFrame(
            columns=["column", "missing_count", "missing_pct", "dtype"]
        )

    out = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "dtype": [str(dtype) for dtype in df.dtypes],
    })

    return out.sort_values(
        ["missing_pct", "column"],
        ascending=[False, True]
    ).reset_index(drop=True)


def add_audit(
    sport,
    test_name,
    source,
    status,
    http_status=None,
    rows=None,
    columns=None,
    notes="",
):
    audit_rows.append({
        "sport": sport,
        "test_name": test_name,
        "source": source,
        "status": status,
        "http_status": http_status,
        "rows_returned": rows,
        "columns_returned": columns,
        "notes": notes,
        "tested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })


def test_bdl_endpoint(
    sport,
    test_name,
    url,
    *,
    params=None,
    required_columns=None,
    show_rows=5,
):
    # Test one BDL endpoint without making an access limitation fatal.
    if not BDL_API_KEY:
        add_audit(
            sport,
            test_name,
            url,
            "SKIPPED",
            notes="BALLDONTLIE_API_KEY not available."
        )
        return pd.DataFrame(), None

    response, error = safe_get(
        url,
        params=params,
        headers=BDL_HEADERS,
        use_bdl_rate_limit=True,
    )

    if response is None:
        add_audit(sport, test_name, url, "ERROR", notes=error)
        print(f"❌ {sport} — {test_name}: {error}")
        return pd.DataFrame(), None

    status_code = response.status_code

    if status_code != 200:
        text_preview = response.text[:250].replace("\n", " ")
        access_note = {
            401: "Unauthorized or account tier does not include this endpoint.",
            403: "Forbidden for this account/API tier.",
            404: "Endpoint or resource not found.",
            429: "Rate limit reached.",
        }.get(status_code, f"HTTP {status_code}")

        add_audit(
            sport,
            test_name,
            url,
            "LIMITED",
            http_status=status_code,
            notes=f"{access_note} Response: {text_preview}",
        )

        print(f"⚠️ {sport} — {test_name}: {access_note}")
        return pd.DataFrame(), response

    try:
        payload = response.json()
    except ValueError:
        add_audit(
            sport,
            test_name,
            url,
            "ERROR",
            http_status=status_code,
            notes="Response was not valid JSON.",
        )
        return pd.DataFrame(), response

    df = json_data_to_frame(payload)

    missing_required = []
    if required_columns:
        missing_required = [col for col in required_columns if col not in df.columns]

    status = "PASS" if not missing_required and not df.empty else "PARTIAL"

    note_parts = []
    if df.empty:
        note_parts.append("Endpoint returned no rows for the requested test.")
    if missing_required:
        note_parts.append(f"Missing expected columns: {missing_required}")

    meta = payload.get("meta", {}) if isinstance(payload, dict) else {}
    if meta:
        note_parts.append(f"meta={meta}")

    add_audit(
        sport,
        test_name,
        url,
        status,
        http_status=status_code,
        rows=len(df),
        columns=len(df.columns),
        notes=" | ".join(note_parts),
    )

    print(
        f"{'✅' if status == 'PASS' else '⚠️'} "
        f"{sport} — {test_name}: {status} "
        f"({len(df)} rows, {len(df.columns)} columns)"
    )

    if not df.empty and show_rows:
        display(df.head(show_rows))

    return df, response

# ============================================================
# 5. NBA — teams
# ============================================================

nba_teams_raw, _ = test_bdl_endpoint(
    sport="NBA",
    test_name="teams",
    url=f"{NBA_BASE}/v1/teams",
    required_columns=[
        "id",
        "conference",
        "division",
        "city",
        "name",
        "full_name",
        "abbreviation",
    ],
)

if not nba_teams_raw.empty:
    nba_teams = (
        nba_teams_raw[
            nba_teams_raw["abbreviation"].isin(NBA_CURRENT_ABBRS)
        ]
        .drop_duplicates(subset="abbreviation")
        .sort_values("full_name")
        .reset_index(drop=True)
    )

    print(f"\nCurrent NBA teams retained: {len(nba_teams)} / 30")
    display(
        nba_teams[
            ["id", "conference", "division", "city", "name", "full_name", "abbreviation"]
        ]
    )

    if len(nba_teams) == 30:
        print("✅ NBA current-team check passed.")
    else:
        print(
            "⚠️ NBA current-team check did not return exactly 30. "
            "Review team abbreviations/API response before Week 3."
        )
else:
    nba_teams = pd.DataFrame()

# ============================================================
# 6. NBA — game/date/score test using one completed season
# ============================================================

NBA_TEST_SEASON = 2025

nba_games_sample, _ = test_bdl_endpoint(
    sport="NBA",
    test_name=f"games sample — {NBA_TEST_SEASON}",
    url=f"{NBA_BASE}/v1/games",
    params={
        "seasons[]": NBA_TEST_SEASON,
        "season_type": "regular",
        "per_page": 100,
    },
    required_columns=[
        "id",
        "date",
        "season",
        "status",
        "home_team_score",
        "visitor_team_score",
        "home_team.id",
        "home_team.full_name",
        "visitor_team.id",
        "visitor_team.full_name",
    ],
)

if not nba_games_sample.empty:
    nba_game_view_cols = [
        col for col in [
            "id", "date", "datetime", "season", "status", "status_state",
            "home_team.full_name", "visitor_team.full_name",
            "home_team_score", "visitor_team_score",
        ]
        if col in nba_games_sample.columns
    ]

    display(nba_games_sample[nba_game_view_cols].head(10))

    nba_missing = missingness_table(nba_games_sample)
    print("\nNBA sample missingness:")
    display(nba_missing.head(20))
else:
    nba_missing = pd.DataFrame()

# ============================================================
# 7. NBA — verify planned historical seasons exist
# ============================================================

nba_season_probe_rows = []

for season in NBA_SEASONS:
    if not BDL_API_KEY:
        nba_season_probe_rows.append({
            "sport": "NBA",
            "season": season,
            "available": None,
            "http_status": None,
            "sample_game_id": None,
            "sample_date": None,
            "note": "Skipped: API key unavailable",
        })
        continue

    response, error = safe_get(
        f"{NBA_BASE}/v1/games",
        params={
            "seasons[]": season,
            "season_type": "regular",
            "per_page": 1,
        },
        headers=BDL_HEADERS,
        use_bdl_rate_limit=True,
    )

    if response is None:
        nba_season_probe_rows.append({
            "sport": "NBA",
            "season": season,
            "available": False,
            "http_status": None,
            "sample_game_id": None,
            "sample_date": None,
            "note": error,
        })
        continue

    if response.status_code == 200:
        payload = response.json()
        data = payload.get("data", [])
        available = len(data) > 0
        first = data[0] if available else {}

        nba_season_probe_rows.append({
            "sport": "NBA",
            "season": season,
            "available": available,
            "http_status": 200,
            "sample_game_id": first.get("id"),
            "sample_date": first.get("date"),
            "note": "",
        })
    else:
        nba_season_probe_rows.append({
            "sport": "NBA",
            "season": season,
            "available": False,
            "http_status": response.status_code,
            "sample_game_id": None,
            "sample_date": None,
            "note": response.text[:150],
        })

nba_season_probe = pd.DataFrame(nba_season_probe_rows)
display(nba_season_probe)

# ============================================================
# 8. NBA — test optional team season averages endpoint
# ============================================================
# This endpoint may require a higher BALLDONTLIE account tier.
# Failure here does NOT invalidate the project because Week 6 can derive
# rolling win %, points scored/allowed, point differential, rest, and Elo
# from game-level results.

nba_team_stats_sample, _ = test_bdl_endpoint(
    sport="NBA",
    test_name="team season averages — optional access test",
    url=f"{NBA_BASE}/nba/v1/team_season_averages/general",
    params={
        "season": 2025,
        "season_type": "regular",
        "type": "base",
        "per_page": 5,
    },
    required_columns=[],
)

if not nba_team_stats_sample.empty:
    print("NBA optional team-stat columns:")
    print(nba_team_stats_sample.columns.tolist())

# ============================================================
# 9. NFL — teams
# ============================================================

nfl_teams_raw, _ = test_bdl_endpoint(
    sport="NFL",
    test_name="teams",
    url=f"{NFL_BASE}/nfl/v1/teams",
    required_columns=[
        "id",
        "conference",
        "division",
        "location",
        "name",
        "full_name",
        "abbreviation",
    ],
)

if not nfl_teams_raw.empty:
    nfl_teams = (
        nfl_teams_raw[
            nfl_teams_raw["abbreviation"].isin(NFL_CURRENT_ABBRS)
        ]
        .drop_duplicates(subset="abbreviation")
        .sort_values("full_name")
        .reset_index(drop=True)
    )

    print(f"\nCurrent NFL teams retained: {len(nfl_teams)} / 32")
    display(
        nfl_teams[
            ["id", "conference", "division", "location", "name", "full_name", "abbreviation"]
        ]
    )

    if len(nfl_teams) == 32:
        print("✅ NFL current-team check passed.")
    else:
        print(
            "⚠️ NFL current-team check did not return exactly 32. "
            "Review team abbreviations/API response before Week 4."
        )
else:
    nfl_teams = pd.DataFrame()

# ============================================================
# 10. NFL — game/date/score test using one completed season
# ============================================================

NFL_TEST_SEASON = 2025

nfl_games_sample, _ = test_bdl_endpoint(
    sport="NFL",
    test_name=f"games sample — {NFL_TEST_SEASON}",
    url=f"{NFL_BASE}/nfl/v1/games",
    params={
        "seasons[]": NFL_TEST_SEASON,
        "season_types[]": 2,
        "per_page": 100,
    },
    required_columns=[
        "id",
        "date",
        "season",
        "week",
        "status",
        "home_team_score",
        "visitor_team_score",
        "home_team.id",
        "home_team.full_name",
        "visitor_team.id",
        "visitor_team.full_name",
    ],
)

if not nfl_games_sample.empty:
    nfl_game_view_cols = [
        col for col in [
            "id", "date", "season", "week", "status", "status_state",
            "home_team.full_name", "visitor_team.full_name",
            "home_team_score", "visitor_team_score",
        ]
        if col in nfl_games_sample.columns
    ]

    display(nfl_games_sample[nfl_game_view_cols].head(10))

    nfl_missing = missingness_table(nfl_games_sample)
    print("\nNFL sample missingness:")
    display(nfl_missing.head(20))
else:
    nfl_missing = pd.DataFrame()

# ============================================================
# 11. NFL — verify planned historical seasons exist
# ============================================================

nfl_season_probe_rows = []

for season in NFL_SEASONS:
    if not BDL_API_KEY:
        nfl_season_probe_rows.append({
            "sport": "NFL",
            "season": season,
            "available": None,
            "http_status": None,
            "sample_game_id": None,
            "sample_date": None,
            "note": "Skipped: API key unavailable",
        })
        continue

    response, error = safe_get(
        f"{NFL_BASE}/nfl/v1/games",
        params={
            "seasons[]": season,
            "season_types[]": 2,
            "per_page": 1,
        },
        headers=BDL_HEADERS,
        use_bdl_rate_limit=True,
    )

    if response is None:
        nfl_season_probe_rows.append({
            "sport": "NFL",
            "season": season,
            "available": False,
            "http_status": None,
            "sample_game_id": None,
            "sample_date": None,
            "note": error,
        })
        continue

    if response.status_code == 200:
        payload = response.json()
        data = payload.get("data", [])
        available = len(data) > 0
        first = data[0] if available else {}

        nfl_season_probe_rows.append({
            "sport": "NFL",
            "season": season,
            "available": available,
            "http_status": 200,
            "sample_game_id": first.get("id"),
            "sample_date": first.get("date"),
            "note": "",
        })
    else:
        nfl_season_probe_rows.append({
            "sport": "NFL",
            "season": season,
            "available": False,
            "http_status": response.status_code,
            "sample_game_id": None,
            "sample_date": None,
            "note": response.text[:150],
        })

nfl_season_probe = pd.DataFrame(nfl_season_probe_rows)
display(nfl_season_probe)

# ============================================================
# 12. NFL — test team-season statistics
# ============================================================
# The endpoint requires team_ids, so use one returned NFL team when available.
# If account access is limited, the notebook records that limitation instead
# of failing Week 2.

if not nfl_teams.empty:
    nfl_test_team_id = int(nfl_teams.iloc[0]["id"])
else:
    nfl_test_team_id = 1

nfl_team_stats_sample, _ = test_bdl_endpoint(
    sport="NFL",
    test_name="team season stats — access test",
    url=f"{NFL_BASE}/nfl/v1/team_season_stats",
    params={
        "season": 2025,
        "team_ids[]": nfl_test_team_id,
        "season_types[]": 2,
        "per_page": 5,
    },
    required_columns=[],
)

if not nfl_team_stats_sample.empty:
    print("NFL team-stat columns:")
    print(nfl_team_stats_sample.columns.tolist())

# ============================================================
# 13. Premier League — helper for Football-Data.co.uk CSV files
# ============================================================

def epl_csv_url(code):
    return f"{FOOTBALL_DATA_BASE}/{code}/E0.csv"


def load_football_data_csv(code):
    url = epl_csv_url(code)

    response, error = safe_get(url)

    if response is None:
        return pd.DataFrame(), url, error, None

    if response.status_code != 200:
        return (
            pd.DataFrame(),
            url,
            f"HTTP {response.status_code}: {response.text[:150]}",
            response.status_code,
        )

    from io import BytesIO

    try:
        df = pd.read_csv(BytesIO(response.content))
    except Exception as exc:
        return pd.DataFrame(), url, f"CSV parse error: {exc}", response.status_code

    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all").reset_index(drop=True)

    return df, url, None, response.status_code

# ============================================================
# 14. Premier League — test one completed season
# ============================================================

EPL_TEST_LABEL = "2025-26"
EPL_TEST_CODE = EPL_SEASONS[EPL_TEST_LABEL]

epl_sample, epl_test_url, epl_error, epl_http_status = load_football_data_csv(
    EPL_TEST_CODE
)

EPL_REQUIRED_COLUMNS = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

if epl_error:
    add_audit(
        "Premier League",
        f"games/match stats — {EPL_TEST_LABEL}",
        epl_test_url,
        "ERROR",
        http_status=epl_http_status,
        notes=epl_error,
    )
    print("❌ Premier League sample test failed:", epl_error)
else:
    missing_required = [
        col for col in EPL_REQUIRED_COLUMNS if col not in epl_sample.columns
    ]

    status = "PASS" if not missing_required and not epl_sample.empty else "PARTIAL"

    add_audit(
        "Premier League",
        f"games/match stats — {EPL_TEST_LABEL}",
        epl_test_url,
        status,
        http_status=epl_http_status,
        rows=len(epl_sample),
        columns=len(epl_sample.columns),
        notes=(
            f"Missing expected columns: {missing_required}"
            if missing_required else ""
        ),
    )

    print(
        f"{'✅' if status == 'PASS' else '⚠️'} "
        f"Premier League — {EPL_TEST_LABEL}: {status} "
        f"({len(epl_sample)} matches, {len(epl_sample.columns)} columns)"
    )

    preview_cols = [
        col for col in [
            "Date", "HomeTeam", "AwayTeam",
            "FTHG", "FTAG", "FTR",
            "HS", "AS", "HST", "AST",
        ]
        if col in epl_sample.columns
    ]

    display(epl_sample[preview_cols].head(10))

    epl_missing = missingness_table(epl_sample)
    print("\nPremier League sample missingness:")
    display(epl_missing.head(25))

# ============================================================
# 15. Premier League — verify all planned seasons
# ============================================================

epl_season_probe_rows = []
epl_season_frames = {}

for season_label, code in EPL_SEASONS.items():
    df, url, error, status_code = load_football_data_csv(code)

    epl_season_probe_rows.append({
        "sport": "Premier League",
        "season": season_label,
        "code": code,
        "available": bool(not df.empty and error is None),
        "http_status": status_code,
        "matches_returned": len(df),
        "columns_returned": len(df.columns),
        "source_url": url,
        "note": error or "",
    })

    if error is None and not df.empty:
        epl_season_frames[season_label] = df

epl_season_probe = pd.DataFrame(epl_season_probe_rows)
display(epl_season_probe)

# ============================================================
# 16. Premier League — teams present in each selected season
# ============================================================

epl_team_rows = []

for season_label, df in epl_season_frames.items():
    if {"HomeTeam", "AwayTeam"}.issubset(df.columns):
        teams = sorted(
            set(df["HomeTeam"].dropna()).union(set(df["AwayTeam"].dropna()))
        )

        for team in teams:
            epl_team_rows.append({
                "season": season_label,
                "team": team,
            })

epl_teams_by_season = pd.DataFrame(epl_team_rows)

if not epl_teams_by_season.empty:
    epl_team_counts = (
        epl_teams_by_season
        .groupby("season", as_index=False)["team"]
        .nunique()
        .rename(columns={"team": "unique_teams"})
    )

    print("Premier League team count by season:")
    display(epl_team_counts)

    print("Teams in the most recent completed season:")
    display(
        epl_teams_by_season[
            epl_teams_by_season["season"] == "2025-26"
        ].reset_index(drop=True)
    )

# ============================================================
# 17. Define the variables needed by the capstone draft
# ============================================================

planned_variables = pd.DataFrame([
    ["game_id", "ALL", "identifier", "required", "source/derived"],
    ["game_date", "ALL", "time", "required", "source"],
    ["home_team", "ALL", "identifier", "required", "source"],
    ["away_team", "ALL", "identifier", "required", "source"],
    ["home_score", "ALL", "outcome", "required", "source after game"],
    ["away_score", "ALL", "outcome", "required", "source after game"],
    ["home_win", "ALL", "outcome", "required", "derived after game"],
    ["win_percentage", "ALL", "team strength", "required", "derive chronologically"],
    ["point_or_goal_differential", "ALL", "team strength", "required", "derive chronologically"],
    ["elo_rating", "ALL", "team strength", "required", "derive chronologically"],
    ["recent_form_3", "ALL", "recent form", "required", "derive chronologically"],
    ["recent_form_5", "ALL", "recent form", "required", "derive chronologically"],
    ["recent_form_10", "ALL", "recent form", "optional", "derive chronologically"],
    ["days_rest", "NBA/NFL", "context", "required", "derive from dates"],
    ["offensive_rating", "NBA", "offense", "preferred", "advanced endpoint or later derivation"],
    ["defensive_rating", "NBA", "defense", "preferred", "advanced endpoint or later derivation"],
    ["effective_fg_pct", "NBA", "offense", "preferred", "advanced endpoint if accessible"],
    ["three_point_pct", "NBA", "offense", "preferred", "advanced endpoint if accessible"],
    ["yards_per_play", "NFL", "offense", "preferred", "team stats endpoint if accessible"],
    ["passing_efficiency", "NFL", "offense", "preferred", "team stats endpoint if accessible"],
    ["rushing_efficiency", "NFL", "offense", "preferred", "team stats endpoint if accessible"],
    ["red_zone_efficiency", "NFL", "offense", "optional", "only if reliable source available"],
    ["takeaways", "NFL", "defense", "preferred", "team stats endpoint if accessible"],
    ["sack_rate", "NFL", "defense", "preferred", "team stats endpoint if accessible"],
    ["goals", "Premier League", "offense", "required", "source/derive"],
    ["goals_conceded", "Premier League", "defense", "required", "source/derive"],
    ["shots", "Premier League", "offense", "preferred", "source if present"],
    ["shots_on_target", "Premier League", "offense", "preferred", "source if present"],
    ["expected_goals", "Premier League", "offense", "optional", "not guaranteed in Football-Data CSV"],
    ["home_advantage", "ALL", "context", "required", "source/derive"],
    ["opponent_strength", "ALL", "context", "required", "derive chronologically"],
    ["player_availability", "ALL", "context", "optional", "only if reliable pre-game history exists"],
], columns=[
    "variable",
    "sport",
    "category",
    "priority",
    "planned_source",
])

display(planned_variables)

# ============================================================
# 18. Pre-game usability rules
# ============================================================

field_safety_rules = pd.DataFrame([
    ["game date/time", "YES", "Known before game", "Can be used directly."],
    ["home/away team identity", "YES", "Known before game", "Can be used directly."],
    ["venue/home indicator", "YES", "Known before game", "Can be used directly if available."],
    ["current-game final score", "NO", "Created during game", "Outcome only; never a predictor for that same game."],
    ["current-game box score", "NO", "Created during game", "Never use for that same game's prediction."],
    ["previous-game scores", "YES", "Historical before next game", "Use shift(1) before rolling features."],
    ["previous-game match stats", "YES", "Historical before next game", "Use shift(1) before rolling features."],
    ["rolling win percentage", "YES", "Derived from earlier games", "Must be calculated chronologically."],
    ["rolling point/goal differential", "YES", "Derived from earlier games", "Must be calculated chronologically."],
    ["pre-game Elo", "YES", "Updated after prior game", "Save rating before current-game update."],
    ["days of rest", "YES", "Derived from schedule dates", "Use prior game date only."],
    ["injury/player availability", "CONDITIONAL", "May be known before game", "Use only if historical timestamp proves it was available pre-game."],
    ["closing/post-game information", "NO", "May incorporate late or post-start info", "Exclude unless timestamp guarantees pre-game availability."],
], columns=[
    "field_type",
    "pre_game_usable",
    "reason",
    "implementation_rule",
])

display(field_safety_rules)

# ============================================================
# 19. Check whether requested soccer match-stat fields are present
# ============================================================

soccer_field_map = pd.DataFrame([
    ["Date", "match date"],
    ["HomeTeam", "home team"],
    ["AwayTeam", "away team"],
    ["FTHG", "full-time home goals"],
    ["FTAG", "full-time away goals"],
    ["FTR", "full-time result"],
    ["HS", "home shots"],
    ["AS", "away shots"],
    ["HST", "home shots on target"],
    ["AST", "away shots on target"],
    ["HF", "home fouls"],
    ["AF", "away fouls"],
    ["HC", "home corners"],
    ["AC", "away corners"],
], columns=["column", "meaning"])

if not epl_sample.empty:
    soccer_field_map["available_in_test_file"] = soccer_field_map["column"].isin(
        epl_sample.columns
    )
else:
    soccer_field_map["available_in_test_file"] = False

display(soccer_field_map)

# ============================================================
# 20. API/source limitations documented for Week 2
# ============================================================

source_limitations = pd.DataFrame([
    {
        "sport": "NBA",
        "source": "BALLDONTLIE NBA API",
        "limitation": "API key required; some advanced/team-stat endpoints depend on account tier.",
        "project_response": (
            "Use teams/games as the required core source. "
            "Derive rolling win %, scoring, points allowed, point differential, rest, and Elo "
            "from chronological game data. Treat advanced NBA metrics as preferred rather than mandatory."
        ),
    },
    {
        "sport": "NFL",
        "source": "BALLDONTLIE NFL API",
        "limitation": (
            "API key required; detailed team/player/availability endpoints may have tier or historical-coverage limits."
        ),
        "project_response": (
            "Use teams/games as the required core source. "
            "Use team-season statistics only when access and historical coverage are confirmed. "
            "Core form, scoring, rest, strength, and Elo variables can be derived from games."
        ),
    },
    {
        "sport": "Premier League",
        "source": "Football-Data.co.uk",
        "limitation": (
            "CSV schemas may vary by season and expected-goals/player-availability fields are not guaranteed."
        ),
        "project_response": (
            "Use only columns consistently present across selected seasons. "
            "Goals, shots, shots on target, dates, teams, and results are tested explicitly. "
            "Treat xG and player availability as optional."
        ),
    },
    {
        "sport": "ALL",
        "source": "All historical sources",
        "limitation": (
            "Final scores and match/game statistics are post-game information."
        ),
        "project_response": (
            "They may be used only to build historical rolling features for later games. "
            "Feature engineering must shift prior to rolling calculations to prevent leakage."
        ),
    },
])

display(source_limitations)

# ============================================================
# 21. Final season and train/validation/test plan
# ============================================================

season_plan = pd.DataFrame([
    {
        "sport": "NBA",
        "included_seasons": "2021, 2022, 2023, 2024, 2025",
        "display_seasons": "2021-22 through 2025-26",
        "training_period": "2021-22 through 2023-24",
        "validation_period": "2024-25",
        "test_period": "2025-26",
        "reason": "Five recent completed seasons; chronological holdout keeps future games out of training.",
    },
    {
        "sport": "NFL",
        "included_seasons": "2021, 2022, 2023, 2024, 2025",
        "display_seasons": "2021 through 2025",
        "training_period": "2021 through 2023",
        "validation_period": "2024",
        "test_period": "2025",
        "reason": "Five recent completed seasons; chronological holdout preserves realistic prediction order.",
    },
    {
        "sport": "Premier League",
        "included_seasons": "2122, 2223, 2324, 2425, 2526",
        "display_seasons": "2021-22 through 2025-26",
        "training_period": "2021-22 through 2023-24",
        "validation_period": "2024-25",
        "test_period": "2025-26",
        "reason": "Five recent completed seasons with a consistent historical CSV workflow.",
    },
])

display(season_plan)

# ============================================================
# 22. Build Week 2 audit summary
# ============================================================

audit_df = pd.DataFrame(audit_rows)

if audit_df.empty:
    audit_df = pd.DataFrame(columns=[
        "sport", "test_name", "source", "status", "http_status",
        "rows_returned", "columns_returned", "notes", "tested_at_utc"
    ])

print("Week 2 source-test audit:")
display(audit_df)

status_counts = (
    audit_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)
display(status_counts)

# ============================================================
# 23. Create missing-variable assessment
# ============================================================

missing_variable_rows = []

if not nba_games_sample.empty:
    for col in [
        "date", "home_team_score", "visitor_team_score",
        "home_team.full_name", "visitor_team.full_name"
    ]:
        missing_variable_rows.append({
            "sport": "NBA",
            "field": col,
            "present": col in nba_games_sample.columns,
            "missing_pct_if_present": (
                round(float(nba_games_sample[col].isna().mean() * 100), 2)
                if col in nba_games_sample.columns else None
            ),
        })

if not nfl_games_sample.empty:
    for col in [
        "date", "week", "home_team_score", "visitor_team_score",
        "home_team.full_name", "visitor_team.full_name"
    ]:
        missing_variable_rows.append({
            "sport": "NFL",
            "field": col,
            "present": col in nfl_games_sample.columns,
            "missing_pct_if_present": (
                round(float(nfl_games_sample[col].isna().mean() * 100), 2)
                if col in nfl_games_sample.columns else None
            ),
        })

if not epl_sample.empty:
    for col in [
        "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
        "HS", "AS", "HST", "AST"
    ]:
        missing_variable_rows.append({
            "sport": "Premier League",
            "field": col,
            "present": col in epl_sample.columns,
            "missing_pct_if_present": (
                round(float(epl_sample[col].isna().mean() * 100), 2)
                if col in epl_sample.columns else None
            ),
        })

missing_variable_assessment = pd.DataFrame(missing_variable_rows)
display(missing_variable_assessment)

# ============================================================
# 24. Save Week 2 documentation/results
# ============================================================

DOCS_DIR = PROJECT_ROOT / "docs"
RESULTS_DIR = PROJECT_ROOT / "results"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

files_written = []

def save_csv(df, path):
    df.to_csv(path, index=False)
    files_written.append(path)
    print(f"✅ Saved: {path}")


save_csv(audit_df, DOCS_DIR / "week2_source_test_audit.csv")
save_csv(source_limitations, DOCS_DIR / "week2_source_limitations.csv")
save_csv(season_plan, DOCS_DIR / "week2_season_plan.csv")
save_csv(planned_variables, DOCS_DIR / "week2_planned_variables.csv")
save_csv(field_safety_rules, DOCS_DIR / "week2_pre_game_field_rules.csv")
save_csv(missing_variable_assessment, DOCS_DIR / "week2_missing_variable_assessment.csv")
save_csv(epl_season_probe, DOCS_DIR / "week2_epl_season_probe.csv")
save_csv(nba_season_probe, DOCS_DIR / "week2_nba_season_probe.csv")
save_csv(nfl_season_probe, DOCS_DIR / "week2_nfl_season_probe.csv")

if not epl_teams_by_season.empty:
    save_csv(epl_teams_by_season, DOCS_DIR / "week2_epl_teams_by_season.csv")

# ============================================================
# 25. Save human-readable data-source plan (Markdown)
# ============================================================

def markdown_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_csv(index=False)


source_plan_md = f'''# Against All Odds — Week 2 Data-Source Plan

Generated: {datetime.now(timezone.utc).isoformat(timespec="seconds")}

## Required sports

- NBA
- NFL
- Premier League

## Selected historical seasons

{markdown_table(season_plan)}

## Source limitations

{markdown_table(source_limitations)}

## Source-test audit

{markdown_table(audit_df)}

## Pre-game data rule

Only information that would have been known before the target game begins may be used
as a predictor. Final scores and box/match statistics from an earlier game may be used for
a later game only after chronological shifting. Rolling calculations in Week 6 must use
`shift(1)` before the rolling window.

## Week 2 conclusion

The required core collection plan is:

1. NBA: BALLDONTLIE teams + games; derive core chronological team features.
2. NFL: BALLDONTLIE teams + games; use team-stat endpoints only when accessible and reliable.
3. Premier League: Football-Data.co.uk season CSV files; retain fields consistently available
   across the selected seasons.
4. Treat expected goals and historical player availability as optional unless a reliable,
   timestamped source is confirmed.
'''

source_plan_path = DOCS_DIR / "week2_data_source_plan.md"
source_plan_path.write_text(source_plan_md, encoding="utf-8")
files_written.append(source_plan_path)

print(f"✅ Saved: {source_plan_path}")

# ============================================================
# 26. Optional: save small source-test samples
# ============================================================

if not nba_games_sample.empty:
    path = PROJECT_ROOT / "data" / "raw" / "nba" / "week2_nba_games_sample.csv"
    nba_games_sample.to_csv(path, index=False)
    files_written.append(path)
    print(f"✅ Saved: {path}")

if not nfl_games_sample.empty:
    path = PROJECT_ROOT / "data" / "raw" / "nfl" / "week2_nfl_games_sample.csv"
    nfl_games_sample.to_csv(path, index=False)
    files_written.append(path)
    print(f"✅ Saved: {path}")

if not epl_sample.empty:
    path = PROJECT_ROOT / "data" / "raw" / "soccer" / "week2_epl_2025_26_sample.csv"
    epl_sample.to_csv(path, index=False)
    files_written.append(path)
    print(f"✅ Saved: {path}")

# ============================================================
# 27. Week 2 completion checks
# ============================================================

checks = {
    "project_root_exists": PROJECT_ROOT.exists(),
    "docs_directory_exists": DOCS_DIR.exists(),
    "season_plan_created": len(season_plan) == 3,
    "source_limitations_documented": len(source_limitations) >= 3,
    "pre_game_rules_documented": len(field_safety_rules) >= 1,
    "premier_league_seasons_tested": len(epl_season_probe) == len(EPL_SEASONS),
    "nba_season_plan_contains_5_seasons": len(NBA_SEASONS) == 5,
    "nfl_season_plan_contains_5_seasons": len(NFL_SEASONS) == 5,
    "source_plan_markdown_exists": source_plan_path.exists(),
}

completion_df = pd.DataFrame(
    [{"check": k, "passed": bool(v)} for k, v in checks.items()]
)

display(completion_df)

core_pass = completion_df["passed"].all()

print("=" * 78)
print("AGAINST ALL ODDS — WEEK 2 DATA SOURCE TESTING")
print("=" * 78)

if core_pass:
    print("✅ WEEK 2 STRUCTURAL DELIVERABLES COMPLETE")
else:
    print("❌ One or more Week 2 structural checks failed.")

if not BDL_API_KEY:
    print(
        "\n⚠️ NBA/NFL live API verification is still pending because "
        "BALLDONTLIE_API_KEY was not available."
    )
else:
    print("\n✅ BALLDONTLIE live API tests were attempted and recorded in the audit.")

print("\nFinal planned seasons:")
display(season_plan[["sport", "display_seasons", "training_period", "validation_period", "test_period"]])

print("\nFiles written:")
for path in files_written:
    print(" -", path.relative_to(PROJECT_ROOT))

print("\nNext major task: Week 3 — NBA historical collection and cleaning.")


✅ Against All Odds — Week 2 configuration loaded
NBA seasons: [2021, 2022, 2023, 2024, 2025]
NFL seasons: [2021, 2022, 2023, 2024, 2025]
Premier League seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
✅ Project root: /content/againstallodds
✅ Verified 20 project directories
✅ BALLDONTLIE API key loaded securely.
⚠️ NBA — teams: Unauthorized or account tier does not include this endpoint.
⚠️ NBA — games sample — 2025: Unauthorized or account tier does not include this endpoint.


,sport,season,available,http_status,sample_game_id,sample_date,note
0,NBA,2021,False,401,None,None,Unauthorized
1,NBA,2022,False,401,None,None,Unauthorized
2,NBA,2023,False,401,None,None,Unauthorized
3,NBA,2024,False,401,None,None,Unauthorized
4,NBA,2025,False,401,None,None,Unauthorized


⚠️ NBA — team season averages — optional access test: Unauthorized or account tier does not include this endpoint.
⚠️ NFL — teams: Unauthorized or account tier does not include this endpoint.
⚠️ NFL — games sample — 2025: Unauthorized or account tier does not include this endpoint.


,sport,season,available,http_status,sample_game_id,sample_date,note
0,NFL,2021,False,401,None,None,Unauthorized
1,NFL,2022,False,401,None,None,Unauthorized
2,NFL,2023,False,401,None,None,Unauthorized
3,NFL,2024,False,401,None,None,Unauthorized
4,NFL,2025,False,401,None,None,Unauthorized


⚠️ NFL — team season stats — access test: Unauthorized or account tier does not include this endpoint.
✅ Premier League — 2025-26: PASS (380 matches, 132 columns)


...

✅ Saved: /content/againstallodds/docs/week2_source_test_audit.csv
✅ Saved: /content/againstallodds/docs/week2_source_limitations.csv
✅ Saved: /content/againstallodds/docs/week2_season_plan.csv
✅ Saved: /content/againstallodds/docs/week2_planned_variables.csv
✅ Saved: /content/againstallodds/docs/week2_pre_game_field_rules.csv
✅ Saved: /content/againstallodds/docs/week2_missing_variable_assessment.csv
✅ Saved: /content/againstallodds/docs/week2_epl_season_probe.csv
✅ Saved: /content/againstallodds/docs/week2_nba_season_probe.csv
✅ Saved: /content/againstallodds/docs/week2_nfl_season_probe.csv
✅ Saved: /content/againstallodds/docs/week2_epl_teams_by_season.csv
✅ Saved: /content/againstallodds/docs/week2_data_source_plan.md
✅ Saved: /content/againstallodds/data/raw/soccer/week2_epl_2025_26_sample.csv


,check,passed
0,project_root_exists,True
1,docs_directory_exists,True
2,season_plan_created,True
3,source_limitations_documented,True
4,pre_game_rules_documented,True
5,premier_league_seasons_tested,True
6,nba_season_plan_contains_5_seasons,True
7,nfl_season_plan_contains_5_seasons,True
8,source_plan_markdown_exists,True


AGAINST ALL ODDS — WEEK 2 DATA SOURCE TESTING
✅ WEEK 2 STRUCTURAL DELIVERABLES COMPLETE

✅ BALLDONTLIE live API tests were attempted and recorded in the audit.

Final planned seasons:


,sport,display_seasons,training_period,validation_period,test_period
0,NBA,2021-22 through 2025-26,2021-22 through 2023-24,2024-25,2025-26
1,NFL,2021 through 2025,2021 through 2023,2024,2025
2,Premier League,2021-22 through 2025-26,2021-22 through 2023-24,2024-25,2025-26



Files written:
 - docs/week2_source_test_audit.csv
 - docs/week2_source_limitations.csv
 - docs/week2_season_plan.csv
 - docs/week2_planned_variables.csv
 - docs/week2_pre_game_field_rules.csv
 - docs/week2_missing_variable_assessment.csv
 - docs/week2_epl_season_probe.csv
 - docs/week2_nba_season_probe.csv
 - docs/week2_nfl_season_probe.csv
 - docs/week2_epl_teams_by_season.csv
 - docs/week2_data_source_plan.md
 - data/raw/soccer/week2_epl_2025_26_sample.csv

Next major task: Week 3 — NBA historical collection and cleaning.


In [1]:
from google.colab import userdata
BDL_API_KEY = userdata.get("BALLDONTLIE_API_KEY")